# 03 – Feature engineering : vérification des variables

**Objectif** : comprendre et contrôler les variables créées par `src/features.py`.

**Principe** : on décrit chaque équipe par sa **forme sur ses 5 derniers matchs** (points, buts marqués, buts encaissés),
pour l'équipe à domicile, l'équipe à l'extérieur, et la différence entre les deux.
On n'utilise **que le passé** du match à prédire.

> Prérequis : `python -m src.features` (génère `data/processed/features.csv`).

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config
from src.features import FEATURES, add_rolling_form, load_features, to_team_matches
from src.data_cleaning import load_clean

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)
W = config.ROLLING_WINDOW

## 1. Avant / après

In [2]:
clean = load_clean()
feats = load_features()

print(f"Matchs nettoyés      : {len(clean)}")
print(f"Matchs avec features : {len(feats)}")
print(f"Matchs retirés       : {len(clean) - len(feats)} (équipes sans {W} matchs d'historique)")
feats.head()

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\angel\\Codage\\Build_Project\\Psg-ML\\ML_prono\\data\\processed\\matches_clean.csv'

Les premiers matchs de chaque équipe (et ceux des équipes nouvellement promues) n'ont pas assez
d'historique pour calculer une forme sur 5 matchs : on les retire plutôt que d'inventer des valeurs.

## 2. Vérification anti-fuite (*data leakage*) sur un exemple

In [ ]:
long = add_rolling_form(to_team_matches(clean), W)
team = long[long["team"] == config.FOCUS_TEAM].reset_index(drop=True)

team[["Date", "side", "gf", "ga", "pts", f"pts_last{W}"]].head(10)

In [ ]:
manual = team.loc[0:W - 1, "pts"].mean()
computed = team.loc[W, f"pts_last{W}"]

print(f"Moyenne des points des {W} premiers matchs : {manual:.2f}")
print(f"Feature du match n°{W + 1}                      : {computed:.2f}")
print("Les {} premières lignes sont NaN (pas d'historique) : {}".format(
    W, team[f"pts_last{W}"].head(W).isna().all()))
assert abs(manual - computed) < 1e-9

La feature du 6ᵉ match est bien la moyenne des 5 matchs **précédents**, et le résultat du 6ᵉ match n'y figure pas.
C'est le rôle de `shift(1)` dans `add_rolling_form`.

## 3. Statistiques descriptives des features

In [ ]:
feats[FEATURES].describe().round(2)

In [ ]:
print("Valeurs manquantes :", feats[FEATURES].isna().sum().sum())
print("Cible :", config.TARGET, "| taux de 1 =", f"{feats[config.TARGET].mean():.1%}")

**À vérifier** : points moyens entre 0 et 3, buts moyens plausibles (autour de 1 à 2 par match),
colonnes `diff_*` centrées sur 0 (en moyenne, domicile et extérieur ont des niveaux comparables).

## 4. Les features sont-elles liées à la cible ?

In [ ]:
feats.groupby(config.TARGET)[FEATURES].mean().T.round(2)

In [ ]:
corr = feats[FEATURES + [config.TARGET]].corr()[config.TARGET].drop(config.TARGET).sort_values()

fig, ax = plt.subplots(figsize=(7, 4))
corr.plot(kind="barh", ax=ax)
ax.set_title("Corrélation de chaque feature avec home_win")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.boxplot(data=feats, x=config.TARGET, y=f"diff_pts_last{W}", ax=ax)
ax.set_title("Écart de forme (domicile - extérieur) selon le résultat")
plt.show()

**Lecture** : si les équipes à domicile qui gagnent ont en moyenne une meilleure forme relative que celles qui ne gagnent pas,
les features portent de l'information. On s'attend à des corrélations **faibles** (le football est très aléatoire) :
c'est normal, et c'est pourquoi la baseline sera difficile à battre.

## 5. Corrélations entre features

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(feats[FEATURES].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
plt.show()

Les `diff_*` sont fortement corrélées aux `home_*` et `away_*` dont elles sont calculées : c'est attendu.
Avec une régression logistique, cette redondance n'est pas un problème grave sur un petit nombre de variables.

## 6. Conclusions (à compléter après exécution)

- **Nombre de matchs utilisables** : _à compléter_.
- **Features les plus liées à la cible** : _à compléter_ (quelles corrélations ressortent ?).
- **Limite connue** : la forme est calculée sur tout l'historique de l'équipe, y compris à travers les saisons
  (une équipe redescendue puis remontée garde d'anciens matchs dans sa fenêtre).
- **Pistes d'amélioration** : forme à domicile / à l'extérieur séparée, tirs cadrés, fenêtres de 3 ou 10 matchs, classement.

➡️ **Étape 4** : entraînement (`src/train.py`) avec une baseline, puis une régression logistique.